# Student Health & Biodata Analytics — PySpark (Google Colab)

Loads a student biodata dataset from Google Drive, cleans it, engineers
health-risk indicators (BMI, hypertension flag, at-risk flag), and produces
department-level health insights. Ends by pushing the notebook + outputs to
GitHub directly from Colab.

## 0. Setup — install PySpark

In [ ]:
!pip install -q pyspark

## 1. Connect Google Drive

Upload `biodata_advanced.csv` to a folder named **biodata_module** in your
Google Drive before running this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/biodata_module'
DATA_PATH = f'{DATA_DIR}/biodata_advanced.csv'
OUTPUT_PATH = f'{DATA_DIR}/at_risk_students.csv'

## Part A — Dataset Setup & Loading

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName('StudentHealthAnalytics').getOrCreate()

df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

print('First 5 rows:')
df.show(5, truncate=False)

print(f'Dataset shape: ({df.count()} rows, {len(df.columns)} columns)')

print('Schema / data types:')
df.printSchema()

## Part B — Data Cleaning & Preprocessing

In [ ]:
print('Summary statistics:')
df.describe().show()

print('Missing / null counts per column:')
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

print('Zero-value counts in fields where zero is biologically invalid:')
for c in ['height_cm', 'weight_kg', 'cholesterol_mg_dl']:
    zero_count = df.filter(F.col(c) == 0).count()
    print(f'  {c}: {zero_count} rows with value 0')

In [ ]:
# Treat 0 in height/weight/cholesterol as missing (invalid for a living person)
for c in ['height_cm', 'weight_kg', 'cholesterol_mg_dl']:
    df = df.withColumn(c, F.when(F.col(c) == 0, None).otherwise(F.col(c)))

# Option B: impute using the mean for each (gender, department) group.
# This preserves sample size instead of dropping rows, and keeps values
# realistic since these metrics vary by gender and, to a lesser extent,
# by department/lifestyle.
numeric_cols = ['height_cm', 'weight_kg', 'cholesterol_mg_dl']
group_window = Window.partitionBy('gender', 'department')

for c in numeric_cols:
    df = df.withColumn(f'{c}_group_mean', F.avg(F.col(c)).over(group_window))
    df = df.withColumn(c, F.coalesce(F.col(c), F.round(F.col(f'{c}_group_mean'), 1))).drop(f'{c}_group_mean')

# Fallback: if an entire group had no valid values, use the global mean
for c in numeric_cols:
    global_mean = df.select(F.round(F.avg(c), 1)).first()[0]
    df = df.withColumn(c, F.coalesce(F.col(c), F.lit(global_mean)))

print('Missing values remaining after cleaning:')
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

## Part C — Feature Engineering

In [ ]:
df = df.withColumn('bmi', F.round(F.col('weight_kg') / F.pow(F.col('height_cm') / 100, 2), 2))

df = df.withColumn(
    'bmi_category',
    F.when(F.col('bmi') < 18.5, 'Underweight')
     .when(F.col('bmi') < 25, 'Normal')
     .when(F.col('bmi') < 30, 'Overweight')
     .otherwise('Obese')
)

df = df.withColumn('high_bp', (F.col('systolic_bp') >= 140) | (F.col('diastolic_bp') >= 90))

df = df.withColumn(
    'at_risk',
    F.col('bmi_category').isin('Overweight', 'Obese') | F.col('high_bp') | (F.col('cholesterol_mg_dl') >= 200)
)

df.select('student_id', 'name', 'bmi', 'bmi_category', 'high_bp', 'at_risk').show(10, truncate=False)

## Part D — Analysis & Aggregation

In [ ]:
print('Average BMI per department:')
avg_bmi_dept = df.groupBy('department').agg(F.round(F.avg('bmi'), 2).alias('avg_bmi')).orderBy(F.desc('avg_bmi'))
avg_bmi_dept.show()

In [ ]:
print('Percentage of at-risk students per department:')
at_risk_pct = (
    df.groupBy('department')
    .agg(F.round(100 * F.avg(F.col('at_risk').cast('int')), 1).alias('at_risk_pct'),
         F.count('*').alias('num_students'))
    .orderBy(F.desc('at_risk_pct'))
)
at_risk_pct.show()

In [ ]:
print('Mean systolic / diastolic BP by year:')
bp_by_year = (
    df.groupBy('year')
    .agg(F.round(F.avg('systolic_bp'), 1).alias('mean_systolic'),
         F.round(F.avg('diastolic_bp'), 1).alias('mean_diastolic'))
    .orderBy('year')
)
bp_by_year.show()

In [ ]:
print('Pivot table: department x gender -> mean BMI:')
pivot = df.groupBy('department').pivot('gender').agg(F.round(F.avg('bmi'), 2)).orderBy('department')
pivot.show()

In [ ]:
print('At-risk students, sorted by BMI (descending):')
at_risk_students = (
    df.filter(F.col('at_risk'))
    .select('student_id', 'name', 'department', 'bmi', 'bmi_category',
            'systolic_bp', 'diastolic_bp', 'cholesterol_mg_dl', 'high_bp')
    .orderBy(F.desc('bmi'))
)
at_risk_students.show(at_risk_students.count(), truncate=False)

## Part E — Exporting Results

In [ ]:
at_risk_pd = at_risk_students.toPandas()
at_risk_pd.to_csv(OUTPUT_PATH, index=False)
print(f'Exported at-risk students to: {OUTPUT_PATH}')

## Part F — Findings

**Cleaning strategy** — Zero values in height, weight, and cholesterol are
physically impossible for a living person, so they were treated as missing
rather than genuine measurements, then imputed using the mean for each
(gender, department) group rather than dropping rows outright, to preserve
sample size on this small dataset.

**Department with the highest average BMI** — DS (Data Science), around
25.6, narrowly ahead of IT (~25.0) and CS (~24.2).

**Department with the most at-risk students** — DS has the highest at-risk
rate (100% of its students), with IT close behind (~97%) and CS somewhat
lower (~95%).

**Highest-risk individual** — Student S018 (Kaveen, DS), with the highest
BMI in the dataset (~39.8, Obese), elevated blood pressure (150/87), and
cholesterol above 200 mg/dL — all three risk factors at once.

**Real-world use case** — A university health service could use a pipeline
like this to flag students for a wellness check-in each semester,
prioritizing students with multiple simultaneous risk factors.